In [ ]:
import numpy as np
import pandas as pd
import ogstools as ot

import pyvista as pv
import matplotlib.pyplot as plt

from pathlib import Path
import os
from scipy.spatial import KDTree

In [ ]:
try:
    cwd = Path(__file__).resolve()
except NameError:
    cwd = Path(os.getcwd()).resolve()

mesh = ot.MeshSeries(cwd.parent/"meshing"/'mesh.vtu')
bh10 = pv.read(cwd.parent/"meshing"/'BH10.vtu')
ms=ot.MeshSeries(cwd/"_out"/"Stimtec_DFN.pvd")
field_data= pd.read_csv(cwd.parent/'ogs-project'/'BH10_20180718_40.6_SR.csv',header=0)

In [ ]:

p_top = bh10.points[np.argmin(bh10.points[:, 2])] #smaller z is top of bh10 well
p_bottom = bh10.points[np.argmax(bh10.points[:, 2])] #bigger z is bottom of bh10 well

vec_well = p_bottom - p_top
l_total = np.linalg.norm(vec_well)
unit_vec = vec_well / l_total

md_interest = 40.6
target_xyz = p_top + (unit_vec * md_interest)

#finding nearmesh from reservoir
mesh_0 = mesh[0] #geometry anchor
node_id = mesh_0.find_closest_point(target_xyz)

tree = KDTree(mesh_0.points)
distance, index = tree.query(target_xyz)

print(f"The nearest node in the 3D mesh is at index: {index}")
print(f"Distance to that node: {distance} ")

#getting closest node and probe values or variables (all)
ms_node = {var:ms.values(var)[:,index] for var in ms.point_data.keys()}
ms_probes=ms.probe(points=target_xyz)
time_values = ms.timevalues


In [ ]:
pressure=ot.variables.pressure.replace(output_unit='MPa')

fig0, axs=plt.subplots(figsize=[12,7])

ot.plot.line(ms_probes,'time',pressure,ax=axs,color='k',fontsize=14, label=f'p probe: {md_interest} m')
plt.plot(time_values,ms_node['pressure']/1e6,color='tab:blue',label=f'p node: {md_interest} m')

plt.title("Pressure at point of interest (injection)") 
plt.legend()

plt.show()
